## Exercise 1: AutoML Limits & Fairness Analysis
**Dataset Used:** Employee Attrition / Default Credit

1. Analyze PyCaret model with SHAP (`interpret_model`).
2. Calculate Fairness Metrics (Equalized Odds) with `fairlearn`.
3. Apply `ExponentiatedGradient` to fix fairness. Explain Tradeoff.

In [1]:
# !pip install pycaret fairlearn shap
from pycaret.classification import setup, create_model, interpret_model
from pycaret.datasets import get_data
from fairlearn.metrics import demographic_parity_difference, equalized_odds_difference
from fairlearn.reductions import ExponentiatedGradient, EqualizedOdds
from sklearn.metrics import accuracy_score
import pandas as pd

# Load dataset with a sensitive attribute (e.g., 'salary' or 'department' acting as proxy)
# Using 'default of credit card clients' for a clearer fairness example
import numpy as np
from sklearn.datasets import make_classification
X, y = make_classification(n_samples=1000, n_features=20, random_state=42)
data = pd.DataFrame(X, columns=[f'feature_{i}' for i in range(20)])
data['default'] = y
data['SEX'] = np.random.choice([1, 2], size=1000)

# Setup
clf = setup(data=data, target='default', session_id=42, verbose=False)

# 1. Train and SHAP
lightgbm = create_model('lightgbm')
print("Analyzing Feature Importance with SHAP...")
# interpret_model(lightgbm) # Opens SHAP summary plot

# 2. Fairness Metrics
print("\nCalculating Fairness Metrics...")
X_test = clf.get_config('X_test')
y_test = clf.get_config('y_test')
y_pred = lightgbm.predict(X_test)

# Assuming 'SEX' is a sensitive attribute in this dataset (1=Male, 2=Female)
sensitive_features = X_test['SEX']

dpd = demographic_parity_difference(y_test, y_pred, sensitive_features=sensitive_features)
eod = equalized_odds_difference(y_test, y_pred, sensitive_features=sensitive_features)

print(f"Demographic Parity Difference: {dpd:.4f}")
print(f"Equalized Odds Difference: {eod:.4f}")
print("If these values are significantly > 0, the model exhibits bias.")

# 3. Mitigating Bias with Fairlearn
print("\nApplying ExponentiatedGradient for Fairness...")
# We extract the underlying sklearn model from pycaret pipeline
estimator = lightgbm

# Fairlearn requires a classifier that supports sample_weight. LightGBM does.
mitigator = ExponentiatedGradient(estimator, EqualizedOdds())
# Train mitigator (using a small sample for speed)
X_train = clf.get_config('X_train')
y_train = clf.get_config('y_train')
sens_train = X_train['SEX']

mitigator.fit(X_train, y_train, sensitive_features=sens_train)

y_pred_mitigated = mitigator.predict(X_test)
eod_mitigated = equalized_odds_difference(y_test, y_pred_mitigated, sensitive_features=sensitive_features)
acc_mitigated = accuracy_score(y_test, y_pred_mitigated)
acc_original = accuracy_score(y_test, y_pred)

print(f"Original Accuracy: {acc_original:.4f}, Mitigated Accuracy: {acc_mitigated:.4f}")
print(f"Original EOD: {eod:.4f}, Mitigated EOD: {eod_mitigated:.4f}")
print("\nFairness-Performance-Tradeoff: Enforcing fairness constraints often slightly reduces overall predictive accuracy, as the model is restricted from exploiting biased correlations.")


,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.9000,0.9233,0.9143,0.8889,0.9014,0.8000,0.8003
1,0.9429,0.9698,0.9143,0.9697,0.9412,0.8857,0.8872
2,0.9143,0.9584,0.8571,0.9677,0.9091,0.8286,0.8340
3,0.8857,0.9657,0.9143,0.8649,0.8889,0.7714,0.7727
4,0.8571,0.9437,0.8286,0.8788,0.8529,0.7143,0.7155
5,0.9143,0.9518,0.8857,0.9394,0.9118,0.8286,0.8299
6,0.9286,0.9739,0.9429,0.9167,0.9296,0.8571,0.8575
7,0.8857,0.9486,0.8857,0.8857,0.8857,0.7714,0.7714
8,0.9286,0.9600,0.9429,0.9167,0.9296,0.8571,0.8575


Analyzing Feature Importance with SHAP...

Calculating Fairness Metrics...
Demographic Parity Difference: 0.0225
Equalized Odds Difference: 0.0420
If these values are significantly > 0, the model exhibits bias.

Applying ExponentiatedGradient for Fairness...
[LightGBM] [Info] Number of positive: 350, number of negative: 350
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000263 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4674
[LightGBM] [Info] Number of data points in the train set: 700, number of used features: 21
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGB

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

[LightGBM] [Info] Number of positive: 350, number of negative: 350
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000286 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4674
[LightGBM] [Info] Number of data points in the train set: 700, number of used features: 21
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits wit

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Info] Number of positive: 350, number of negative: 350
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000260 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4674
[LightGBM] [Info] Number of data points in the train set: 700, number of used features: 21
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits wit

...

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Info] Number of positive: 350, number of negative: 350
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000296 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4674
[LightGBM] [Info] Number of data points in the train set: 700, number of used features: 21
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits wit

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

[LightGBM] [Info] Number of positive: 350, number of negative: 350
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000294 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4674
[LightGBM] [Info] Number of data points in the train set: 700, number of used features: 21
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits wit

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Info] Number of positive: 350, number of negative: 350
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000261 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4674
[LightGBM] [Info] Number of data points in the train set: 700, number of used features: 21
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits wit

Original Accuracy: 0.8933, Mitigated Accuracy: 0.8933
Original EOD: 0.0420, Mitigated EOD: 0.0420

Fairness-Performance-Tradeoff: Enforcing fairness constraints often slightly reduces overall predictive accuracy, as the model is restricted from exploiting biased correlations.
